In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

In [ ]:
import math
import keyring
import itertools
from ast import literal_eval

In [ ]:
from suncalc import get_position, get_times
from datetime import datetime as dt
import pytz

In [ ]:
# get base path for data
data_dir = keyring.get_password("msp", "vmt_reduction_dir")

In [ ]:
# read in the data
# df = pd.read_csv(data_dir + "/data_processed/tbi_merged.csv")
df = pd.read_csv("../../data_processing/tbi_cleaned.csv")
df

In [ ]:
# read path geopkg data
# parquet files take an order of magnitude less time to read
car = pd.read_parquet(data_dir + "/Data_Processed/geodata/car_congestion_nogeom.parquet")
bike = pd.read_parquet(data_dir + "/Data_Processed/geodata/bike_lts.parquet", columns=["trip_id", "duration_seconds", "distance_meters", "weight", "distance_meters_1", "distance_meters_2", "distance_meters_3", "distance_meters_4"])
transit = gpd.read_parquet(data_dir + "/Data_Processed/geodata/transit_trips.parquet")
walk = pd.read_parquet(data_dir + "/Data_Processed/geodata/walk_trips_nogeom.parquet")
bike = bike.rename(columns={"weight": "duration_seconds", "duration_seconds": "weight"})

In [ ]:
# transit preprocessing

In [ ]:
# project to US equidistant projection to calculate lengths of transit trips (in meters)
# https://spatialreference.org/ref/esri/usa-contiguous-equidistant-conic/
transit = transit.to_crs("ESRI:102005")
transit["length"] = transit.length
transit["access_length"] = transit["length"]
transit

In [ ]:
# calculate duration from start/eend times
transit["start_time_dt"] = pd.to_datetime(transit["start_time"])
transit["end_time_dt"] = pd.to_datetime(transit["end_time"])

transit["duration"] = (transit["end_time_dt"] - transit["start_time_dt"]).apply(lambda x: x.seconds / 60)

In [ ]:
transit["num_transfers"] = (transit["leg_type"] == "TransitRouter.transfer") # temporary field to calculate number of transfers
transit["non_transit_duration"] = (transit["leg_type"].isin(["TransitRouter.access", "TransitRouter.egress"])) * (transit["duration"]) # non-transit time (access and egress)

In [ ]:
# create a gdf for linked trips
agg_fns = {
    "trip_id": "first",
    "leg_index": "count",
    "start_time": "first",
    "end_time": "last",
    "origin_stop_id": "first", # throw away
    "origin_stop_name": "first",
    "destination_stop_id": "first",
    "destination_stop_name": "first",
    "route_id": "first", 
    "route_short_name": "first",
    "route_long_name": "first",
    "route_type": "first", 
    "leg_type": "first", # end throw away
    "start_time_dt": "first",
    "end_time_dt": "last",
    "duration": "sum",
    "length": "sum",
    "access_length": "first",
    "num_transfers": "sum",
    "non_transit_duration": "sum"
}

transit_grouped = transit.dissolve(by="trip_id", aggfunc=agg_fns) # merges geometries in addition to aggregating the rest of the columns

In [ ]:
# 2013 transit trips lack a corresponding path out of 11271 (18%)
# possibly need to relax the optimization restrictions for transfers/other factors
df[(df["mode"] == "Transit") & (~df["trip_id"].isin(transit_grouped["trip_id"]))]

In [ ]:
# many other trips lack a transit trip, likely due to lack of a feasible transit trip
len(df) - len(transit_grouped)

In [ ]:
# generally only longer duration/distance trips tend to not have corresponding trips
display(df[df["mode"] == "Transit"]["duration"].describe() - df[(df["mode"] == "Transit") & (~df["trip_id"].isin(transit_grouped["trip_id"]))]["duration"].describe())
display(df[df["mode"] == "Transit"]["distance"].describe() - df[(df["mode"] == "Transit") & (~df["trip_id"].isin(transit_grouped["trip_id"]))]["distance"].describe())

In [ ]:
# ignore schools bus and taxi/ridehail/carshare
valid_modes = ["Car", "Bike/Scooter", "Walk", "Transit"]

In [ ]:
# plot basic density graph of a single variable
def plot_density(column: pd.Series, percentile=0.95, discrete=False, bins=100, size=(12, 6)):
    fig, ax = plt.subplots(figsize=size)
    sns.histplot(column, ax=ax, discrete=discrete, bins=bins, kde=True, stat="density")
    val = column.quantile(q=percentile)
    plt.axvline(x=val, color="red")
    return fig, ax

### duration analysis

In [ ]:
## overall summary statistics
df["duration"].describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# overall distribution of duration
# there are repetitive peaks, likely since the temporal resolution is discrete with respect ot minutes
fig, ax = plot_density(df["duration"], bins=500)
ax.set_xlim(left=0, right=200) # cutoff upper end, there are some outliers that make the graph hard to read

In [ ]:
# overall distribution of the log of duration
fig, ax = plot_density(np.log(df["duration"]), bins=100)
ax.set_xlabel("log duration")

In [ ]:
## duration analysis segmented by mode

In [ ]:
def plot_mode_density(df: pd.DataFrame, key: str, percentile=0.95, size=(12, 6), bins=300, function=lambda x: x):
    palette = itertools.cycle(sns.color_palette()) # cycle through colors to make sure each mode gets a unique one
    fig, ax = plt.subplots(figsize=size)
    for mode in valid_modes: # cycle through all modes
        c = next(palette) # get color to use
        group = df[df["mode"] == mode] # filter out the current mode
        sns.histplot(function(group[key]), ax=ax, stat="density", kde=True, label=mode, color=c, bins=bins) # plot hist plot with kde overlayed in the color
        val = function(group[key]).quantile(q=percentile) # calculate the value of the given percentile (default 0.95)
        plt.axvline(x=val, color=c) # plot line representing that value on the plot
    return fig, ax

In [ ]:
def show_summaries(df: pd.DataFrame, key: str, percentile=[0.95]): # show normal summaries for each mode side by side
    res = []
    p = [0.25, 0.5, 0.75]
    p = p + percentile
    p = list(set(p))
    # p.append(percentile)
    for mode in valid_modes:
        group = df[df["mode"] == mode]
        res.append(group[key].describe(percentiles=p))
    x = pd.concat(res, axis=1)
    x.columns = valid_modes
    return x

In [ ]:
show_summaries(df, "duration")

In [ ]:
# plot of densities of all modes overlayed on top of one another
fig, ax = plot_mode_density(df, "duration")
ax.set_xlim(left=0, right=150)
plt.legend()

takeaways:

- durations tend to be right-skewed; people prefer shorter durations
- walk durations tend to be the shortest, followed by car, biking, and transit in that order
- there is small bias in durations as it is semi-categorical, with multiple peaks/non-continuous distribution
- the transit duration distribution deviates significantly from the other modes--transit riders are less sensitive to duration
- there remain some duration outliers
- possibly drop duration in favor of distance

potential further analysis:
- trip duration as a percentage of the total tour time

### difference between non-driving trip durations with corresponding driving trip duration

In [ ]:
def get_car_time(row):
    hour = int(row["arrive_time"][0:2])
    sunday = row["travel_dow"] == "Sunday"
    saturday = row["travel_dow"] == "Saturday"
    if sunday:
        query = "sundays"
    elif saturday:
        query = "saturdays"
        query += "_"
    else:
        query = "weekdays"
        query += "_"
    
    
    if hour >= 0 and hour <= 5:
        query += "0-6"
    elif hour >= 20 and hour <= 23:
        query += "20-24"
    else:
        query += str(hour) + "-" + str(hour + 1)
        
    return car.loc[(query, row["trip_id"])]["duration_seconds"] / 60
    

In [ ]:
# overall

difference_with_car_duration = df[(df["mode"] != "Car") & (df["mode"].isin(valid_modes))]["duration"] - \
    df[(df["mode"] != "Car") & (df["mode"].isin(valid_modes))].apply(lambda x: get_car_time(x), axis=1).values

In [ ]:
fig, ax = plot_density(difference_with_car_duration)
ax.set_xlim(left=-50, right=150)

In [ ]:
difference_with_car_duration.describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# percent of all non-car trips that are faster than car
(difference_with_car_duration <= 0).sum() / len(difference_with_car_duration)

In [ ]:
# bike vs car

In [ ]:
# for each bike trip, calculate the corresponding duration it would take to do it via car
# this should tell something about how these bikers tradeoff between car & bike duration
bike_minus_car_duration = df[df["mode"] == "Bike/Scooter"]["duration"] - df[df["mode"] == "Bike/Scooter"].apply(lambda x: get_car_time(x), axis=1).values

In [ ]:
# not sure if it would be feasible to switch to bike if car is 53 minutes faster
# self sampling likely plays a role -- maybe adopt a smaller percentile to adjust?
bike_minus_car_duration.describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
plot_density(bike_minus_car_duration)

In [ ]:
# transit vs car

In [ ]:
# for each transit trip, calculate the corresponding duration it would take to do it via car
# this should tell something about how these transit riders tradeoff between car & transit duration

transit_minus_car_duration = df[df["mode"] == "Transit"]["duration"] - \
    df[df["mode"] == "Transit"].apply(lambda x: get_car_time(x), axis=1).values

In [ ]:
# positive means car duration < transit and vice versa
fig, ax = plot_density(transit_minus_car_duration)
ax.set_xlim(left=-50, right=200)

In [ ]:
transit_minus_car_duration.describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# walk vs car

In [ ]:
# for each walk trip, calculate the corresponding duration it would take to do it via car
# this should tell something about how walkers tradeoff between car & walk duration

walk_minus_car_duration = df[df["mode"] == "Walk"]["duration"] - \
    df[df["mode"] == "Walk"].apply(lambda x: get_car_time(x), axis=1).values

In [ ]:
# positive means car duration < transit and vice versa
fig, ax = plot_density(walk_minus_car_duration)
ax.set_xlim(left=-50, right=150)

In [ ]:
walk_minus_car_duration.describe(percentiles=[0.25, 0.5, 0.75, 0.95])

duration difference takeaways

- the distributions for duration difference between mode and car are remarkably similar, with 95 percnetiles around 40-60
- none of these modes can beat out car (and it doesn't even seem close--the amount of trips in each mode where this mode is faster is like 1% for each), so there are some other factors other than duration that drive these people to use these modes
    - this could be due to selection bias -- people who use these modes are "enthusiasts," so a smaller percentile may be needed to account for this

further analysis
- could try a duration percentage analysis as well to standardize duration magnitudes

### difference between driving trip duration with each mode duration

In [ ]:
# car trip duration - corresponding bike duration

In [ ]:
# for each car trip, find time needed to do it on bike & subtract it from the duration
# - means bike is longer & vice versa
car_duration_minus_bike = df[df["mode"] == "Car"]["duration"] - (bike.set_index("trip_id").loc[df[df["mode"] == "Car"]["trip_id"].values]["weight"] / 60).values

In [ ]:
car_duration_minus_bike.describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# most car trips are faster than corresponding bike ones but they do seem to cluster around 0 (parity)
fig, ax = plot_density(car_duration_minus_bike)
ax.set_xlim(left=-1000, right=500)

In [ ]:
# car trip duration - corresponding walk duration

In [ ]:
# for each car trip, find time needed to do it walking & subtract it from the duration
# - means bike is longer & vice versa
car_duration_minus_walk = df[df["mode"] == "Car"]["duration"] - (walk.set_index("trip_id").loc[df[df["mode"] == "Car"]["trip_id"].values]["duration_seconds"] / 60).values

In [ ]:
car_duration_minus_walk.describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# about the same as bike? but slightly more left-heavy
fig, ax = plot_density(car_duration_minus_walk)
ax.set_xlim(left=-1000, right=500)

In [ ]:
# car trip duration - corresponding transit duration

In [ ]:
# for each car trip, find time needed to do it via transit & subtract it from the duration (if such a time is found)
# - means bike is longer & vice versa
car_duration_minus_transit = df[(df["mode"] == "Car") & (df["trip_id"].isin(transit_grouped["trip_id"]))]["duration"] - (transit_grouped.loc[df[(df["mode"] == "Car") & (df["trip_id"].isin(transit_grouped["trip_id"]))]["trip_id"].values]["duration"]).values

In [ ]:
car_duration_minus_transit.describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# this is significant different from car/biking; much more clustered around 0 aside from a small cluster near -1500
# have to note that some transit times were left out--could be systemic/due to previous filtering
fig, ax = plot_density(car_duration_minus_transit)
ax.set_xlim(left=-2000, right=500)

In [ ]:
# car minus best

In [ ]:
# assume no transit trip means missing and 10000 duration
car_duration_minus_transit_cmp = car_duration_minus_transit.reindex(car_duration_minus_bike.index).fillna(10000)

In [ ]:
alt_mode_duration_best = pd.concat([car_duration_minus_bike, car_duration_minus_transit_cmp, car_duration_minus_walk], axis=1).apply(min, axis=1)

In [ ]:
alt_mode_duration_best.describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# seems to roughly follow transit distribution
fig, ax = plot_density(alt_mode_duration_best)

difference between driving trip duration with each mode duration takeaways

- for car trips, alternative modes are generally faster
- transit appear to be more close to parity, although there is a small bump at -1500 for some reason
- walk/bike are more varied
- generally mirrors the previous section on difference between mode and driving

further analysis

- could do percentages to standardize distance magnitudes

### distance analysis

In [ ]:
# overall

In [ ]:
# summary statistics
df["distance"].describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# distribution
fig, ax = plot_density(df["distance"])
ax.set_xlim(left=0, right=50) # max is 99, cut off about half

In [ ]:
# log distance distribution
# compared with the overall distribution, we see more resolution in the shorter distance trips, which make up a significant portion of all trips
fig, ax = plot_density(np.log(df["distance"]))

In [ ]:
# segmented by mode

In [ ]:
show_summaries(df, key="distance")

In [ ]:
# raw distance distribution (miles)
fig, ax = plot_mode_density(df, key="distance") # walk peak distorts the plot
plt.legend()

In [ ]:
# raw distance distribution (miles) with walk omitted
fig, ax = plot_mode_density(df[df["mode"] != "Walk"], key="distance")
plt.legend()

In [ ]:
# log distance distribution
fig, ax = plot_mode_density(df, key="distance", function=np.log)
plt.legend()

In [ ]:
# car distance distribution
fig, ax = plot_density(df[df["mode"] == "Car"]["distance"])

In [ ]:
# transit distance distribution
fig, ax = plot_density(df[df["mode"] == "Transit"]["distance"])
ax.set_xlim(left=0, right=35)

In [ ]:
# walk distance distribution
fig, ax = plot_density(df[df["mode"] == "Walk"]["distance"], bins=300)
ax.set_xlim(left=0, right=4)

In [ ]:
# bike distnace distribution
fig, ax = plot_density(df[df["mode"] == "Bike/Scooter"]["distance"])
ax.set_xlim(left=0, right=20)

distance takeaways

- pretty straight forward, not anything surprising
- more continuous than duration, has a continuous decline from 0 distance trips, very much right skewed
    - people prefer shorter distance trips
- car and transit distance distributions are relatively similar -- transit & car riders have relatively similar sensitivies to distance
- walk trips are generally about 1 mile or less
- bike/scooter trips are a middle ground between car/transit and walk

further analysis
- we could do a distance vs car analysis but this probably wouldn't show much--difference between walking around a square and cutting through the middle
- maybe distance in certain areas? like cbd vs suburban or something
    - already done with lts + los


### season / temperature

In [ ]:
df["month"] = df["travel_date"].str[5:7]

In [ ]:
def map_to_season(month):
    # can change to a more granular definition
    match month:
        case '12' | '01' | '02':
            return "winter"
        case '03' | '04' | '05':
            return "spring"
        case '06' | '07' | '08':
            return "summer"
        case _:
            return "autumn"

In [ ]:
df["season"] = df["month"].apply(map_to_season)
df["season"] = df["month"].apply(map_to_season)

In [ ]:
# distribution of season
# WARNING: this could be biased due to the timespan of the data collection -- some season may have been cut short
sns.countplot(x="season", data=df)

In [ ]:
# season distribution by mode

In [ ]:
# mode share
def plot_mode_barplot_pct_total(df: pd.DataFrame, key: str, mode:str, func=lambda x: x, figsize=(7, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(x=df[key].value_counts().sort_index().index, y=func(df[df["mode"] == mode][key].value_counts().sort_index()), ax=ax)
    return fig, ax

In [ ]:
def plot_stacked_mode_share(x):
    fig, ax = plt.subplots()
    vals = df[x].value_counts().sort_index().index
    temp = pd.DataFrame(index=["Car", "Transit", "Bike/Scooter", "Walk"], columns=vals, data=0)
    temp.loc["Car"] = df[df["mode"] == "Car"][x].value_counts().sort_index() / df[x].value_counts()
    temp.loc["Walk"] = df[df["mode"] == "Walk"][x].value_counts().sort_index() / df[x].value_counts()
    temp.loc["Transit"] = df[df["mode"] == "Transit"][x].value_counts().sort_index() / df[x].value_counts()
    temp.loc["Bike/Scooter"] = df[df["mode"] == "Bike/Scooter"][x].value_counts().sort_index() / df[x].value_counts()
    print(temp)
    print(df[df["mode"] == "Car"][x].value_counts() / df[x].value_counts())
    temp.T.plot(kind="bar", stacked=True, ax=ax)
    return fig, ax

In [ ]:
# overall mode shares in stacked format to allow for easy reading
fig, ax = plot_stacked_mode_share("season")

In [ ]:
# car seasons
# little variation in car trip % between seasons aside from a slight dip in the summer and slight bump in winter
# thsi represents the % of trips in each season that were by car
fig, ax = plot_mode_barplot_pct_total(df, "season", "Car", lambda x: x / df["season"].value_counts())

In [ ]:
# transit seasons
# significant summer dip & somewhat significant winter bump (possibly due to unsafe conditions otherwise?)
fig, ax = plot_mode_barplot_pct_total(df, "season", "Transit", lambda x: x / df["season"].value_counts())

In [ ]:
# walk seasons
# significant summer bump & significant winter dip
fig, ax = plot_mode_barplot_pct_total(df, "season", "Walk", lambda x: x / df["season"].value_counts())

In [ ]:
# bike season
# very significant summer bump & very significant winter dip
# basically walk but more extreme variation
fig, ax = plot_mode_barplot_pct_total(df, "season", "Bike/Scooter", lambda x: x / df["season"].value_counts())

In [ ]:
# segmentation by month

In [ ]:
# distribution of month
# WARNING: this could be biased due to the timespan of the data collection -- some season may have been cut short
sns.countplot(x="month", data=df)

In [ ]:
# ovearll mode share in terms of month in stacked format
fig, ax = plot_stacked_mode_share("month")

In [ ]:
# car months
fig, ax = plot_mode_barplot_pct_total(df, "month", "Car", lambda x: x / df["month"].value_counts())

In [ ]:
# walk months
fig, ax = plot_mode_barplot_pct_total(df, "month", "Walk", lambda x: x / df["month"].value_counts())

In [ ]:
# bike months
fig, ax = plot_mode_barplot_pct_total(df, "month", "Bike/Scooter", lambda x: x / df["month"].value_counts())

In [ ]:
# transit months
fig, ax = plot_mode_barplot_pct_total(df, "month", "Transit", lambda x: x / df["month"].value_counts())

In [ ]:
# temperature/precipitation/snowfall

In [ ]:
# https://www.ncei.noaa.gov/pub/data/ghcn/daily/
weather = pd.read_csv("extra_data/USW00014922.csv")
weather = weather[["Date", "Measurement", "Value"]]
weather = weather.pivot(index="Date", columns="Measurement", values="Value")
weather

In [ ]:
weather["year"] = weather.index.astype(str).str[0:4]
weather["month"] = weather.index.astype(str).str[4:6]
weather["day"] = weather.index.astype(str).str[6:8]
weather["date"] = weather["year"] + "-" + weather["month"] + "-" + weather["day"]
weather = weather.set_index("date")
weather

In [ ]:
# see https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt for a key
# see https://www.weather.gov/gsp/snow for a more in-depth explanation of snow depth
df["temperature"] = (weather.loc[df["travel_date"].values, "TAVG"] / 10).values # average temperature in Celsius
df["precipitation"] = (weather.loc[df["travel_date"].values, "PRCP"] / 10).values # precipition in mm
df["snowfall"] = (weather.loc[df["travel_date"].values, "SNOW"] / 10).values # snowfall in mm

In [ ]:
df["temperature_max"] = (weather.loc[df["travel_date"].values, "TMAX"] / 10).values # average temperature in Celsius
df["temperature_min"] = (weather.loc[df["travel_date"].values, "TMIN"] / 10).values # average temperature in Celsius
df["precipitation"] = (weather.loc[df["travel_date"].values, "PRCP"] / 10).values # precipition in mm
df["snow_depth"] = (weather.loc[df["travel_date"].values, "SNWD"] / 10).values # snowfall in mm

In [ ]:
# distribution of average temperature on travel day for each mode
fig, ax = plot_mode_density(df, "temperature", percentile=0.05)
plt.legend()

In [ ]:
# distribution of average temperature on travel day for each mode
fig, ax = plot_mode_density(df, "temperature_min", percentile=0.05)
plt.legend()

In [ ]:
# distribution of average temperature on travel day for each mode
fig, ax = plot_mode_density(df, "temperature_max", percentile=0.05)
plt.legend()

In [ ]:
show_summaries(df, "temperature")

In [ ]:
# distribution of precipitation on travel day for each mode
# not very meaningful as most days didn't have precipitation
# the 95th percentiles overlap
fig, ax = plot_mode_density(df[df["precipitation"] != 0], "precipitation")
plt.legend()
ax.set_xlim(left=0, right=15)

In [ ]:
# can see a bit more with summary statistics (mean)
# lower percentiles reach 0 rainfall
show_summaries(df, "precipitation", percentile=[0.8, 0.85, 0.9, 0.95])

In [ ]:
# distribution of snowfall on travel day for each mode
# not very meaningful as most days didn't have snowfall
fig, ax = plot_mode_density(df[df["snowfall"] != 0], "snowfall")
plt.legend()
ax.set_xlim(left=0, right=15)

In [ ]:
# can see a bit more with summary statistics (mean)
# notably, 95% of bike trips occur when there is no snow
show_summaries(df, "snowfall", percentile=[0.8, 0.85, 0.9, 0.95])

In [ ]:
show_summaries(df, "snow_depth", percentile=[0.8, 0.85, 0.9, 0.95])

takeaways

- seasons do seem to play a large part in the relative proportions of mode trips for non-driving modes
    - comparing relative proportions in distance should be able to handle the sampling timing bias
    - this effect is very pronounced for walking and biking
    - this effect is seen more granularly with month segmentation
- temperature, precipitation, and snowfall also have effects and get more at the causes between month/seaseon trends
    - temperature is the most meaningful; bike trips don't happen in snow 95% of the time

future analysis
- how do we apply a 95% rule for categorical variables? -- probably need an alternative feasibility indicator
    - could just cut out extreme dips -- e.g., people don't bike in winter

### daylight hours indicator & departure time

In [ ]:
def plot_multi_barplot(df: pd.DataFrame, x: str, y: str, figsize=(7, 5), order=None):
    # plot normalized bar plot of the values in each mode side by side
    fig, ax = plt.subplots(figsize=figsize)
    if y == "mode": 
        df = df[df["mode"].isin(valid_modes)]
    (df
     .groupby(y)[x]
     .value_counts(normalize=True)
     .mul(100)
     .rename("percent")
     .reset_index()
     .pipe((sns.barplot, "data"), x=x, y="percent", hue=y, ax=ax, order=order)
    )
    
    return fig, ax

In [ ]:
df["datetime"] = df["travel_date"].apply(lambda x: dt.strptime(x, r"%Y-%m-%d"))
df["datetime"] = df["datetime"].astype('M8[D]').astype('O') # somehow converts datetime64[ns] into datetime

In [ ]:
central = pytz.timezone("US/Central")
df[["sunrise", "sunset"]] = pd.DataFrame(get_times(df['datetime'], df["o_lon"], df["o_lat"]))[["sunrise", "sunset"]].applymap(lambda x: x.astimezone(central)) # parsed as utc, convert to central

In [ ]:
# get standardized sunrise_time/sunset_time to allow for direct string comparison 
df["sunrise_time"] = df["sunrise"].dt.hour.astype(str).str.zfill(2) + ":" + df["sunrise"].dt.minute.astype(str).str.zfill(2) + ":" + df["sunrise"].dt.second.astype(str).str.zfill(2)
df["sunset_time"] = df["sunset"].dt.hour.astype(str).str.zfill(2) + ":" + df["sunset"].dt.minute.astype(str).str.zfill(2) + ":" + df["sunset"].dt.second.astype(str).str.zfill(2)

In [ ]:
# string comparison works since 13 > 03--alphabetical is chronological
df["during_daylight"] = (df["depart_time"] > df["sunrise_time"]) & (df["arrive_time"] < df["sunset_time"])

In [ ]:
# distribution of trips w.r.t occurring wthin daylight hours
# majority are within daylight hours
# this doesn't have the problem as with seasons as daylight occurs on a daily basis, which the survye likely captures
sns.countplot(x="during_daylight", data=df)

In [ ]:
# bar plot of the % of trips in a given mode that occur during/not during daylight side by side for all modes
# less relative walk/transit trips at night/more at day
# more relative transit/car trips at night, more at day
plot_multi_barplot(df, "during_daylight", "mode")

In [ ]:
# departure time analysis -- similar to during_daylight but more continuous

In [ ]:
df["depart_time_dt"] = df["depart_time"].apply(lambda x: dt.strptime(x, "%H:%M:%S"))
df["depart_time_dt"]

In [ ]:
# everything is in 1900 but that doesn't matter -- only look at hours
# need an alternative feasibility indicator other than 95% -- possibly both extremes, 2 tailed?
# most trips occur during daytime
plot_density(df["depart_time_dt"])

In [ ]:
# departure time segmented by mode
# not very meaningful, likely due to daytime differences
fig, ax = plot_mode_density(df, key="depart_time_dt")
plt.legend()

In [ ]:
def get_time_from_daylight(row):
    if row["during_daylight"]:
        return 0
    curr_time = row["depart_time_dt"].hour * 60 * 60 + row["depart_time_dt"].minute * 60 + row["depart_time_dt"].second
    sunrise = row["sunrise"].hour * 60 * 60 + row["sunrise"].minute * 60 + row["sunrise"].second
    sunset = row["sunset"].hour * 60 * 60 + row["sunset"].minute * 60 + row["sunset"].second
    return min(abs(curr_time - sunrise), abs(curr_time - sunset))

In [ ]:
df["time_from_daylight"] = df.apply(lambda x: get_time_from_daylight(x), axis=1) # seconds away from daylight hours

In [ ]:
df["time_from_daylight"].describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# plot of distribution of seconds from daylight, excluding the trips that occur during daylight
fig, ax = plot_density(df[df["time_from_daylight"] != 0]["time_from_daylight"])

In [ ]:
show_summaries(df[~df["during_daylight"]], key="time_from_daylight")

In [ ]:
# seconds from daylight segmented by mode
# doesn't really make sense with 95th percetiles; distribution is also a bit iffy
fig, ax = plot_mode_density(df[df["time_from_daylight"] != 0], "time_from_daylight")
plt.legend()

takeaways

- is as expected -- trips shift to quicker/less vulnerable modes at nighttime
- time from daylight as a continuous metric isn't very meaningful
    - transit is the only that dips the most at longer times from daylight, likely affected by service availability

future analysis

- how to apply to feasibility analysis?
- we could make this more continuous
    - e.g., determine a good time from daylight measurement or just a general valid departure time range

### number of passengers

In [ ]:
# overall analysis

In [ ]:
# parsing into integer summary statistics
df[df["num_travelers"] != "Missing"]["num_travelers"].astype(int).describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
df["num_travelers_numeric"] = np.where(df["num_travelers"] == "Missing", -1, df["num_travelers"])
df["num_travelers_numeric"] = df["num_travelers_numeric"].astype(int)

In [ ]:
sns.countplot(x="num_travelers_numeric", data=df[df["num_travelers"] != "Missing"])

In [ ]:
# side-by-side bar plot of the percentage of trips in each mode that have x travelers
# it seems car and walking are the dominant multi-traveler modes
# bike/scooter and transit are generally left-heavier, except for a seemingly random transit peak at 6 travelers
fig, ax = plot_multi_barplot(df[df["num_travelers"] != "Missing"], "num_travelers_numeric", "mode", figsize=(10, 5))

In [ ]:
# transit months
fig, ax = plot_mode_barplot_pct_total(df, "month", "Transit", lambda x: x / df["month"].value_counts())

In [ ]:
# stacked mode share (% of trips in each number of travelers that have a given mode)
# ignore -1; means missing
plot_stacked_mode_share("num_travelers_numeric")

In [ ]:
fig, ax = plot_mode_barplot_pct_total(df, "num_travelers_numeric", "Transit", lambda x: x / df["num_travelers_numeric"].value_counts())
x, y = plot_mode_barplot_pct_total(df, "num_travelers_numeric", "Car", lambda x: x / df["num_travelers_numeric"].value_counts())

In [ ]:
fig, ax = plot_mode_barplot_pct_total(df, "num_travelers_numeric", "Car", lambda x: x / df["num_travelers_numeric"].value_counts())

In [ ]:
# summary statistics of the number of travelers for eahc mode
show_summaries(df[df["num_travelers"] != "Missing"], key="num_travelers_numeric")

takeaways

- transit/biking are modes generally associated with less travelers while walking/driving are hte opposite
    - there are deviations from this trend -- e.g., the dominance of transit from 6-9 travelers (college students? or groups of people going somewhere without cars)

future analysis
- 95% rule probably won't work here -- it's not really reasonable to assume anybody with 4 travelers can just ride via bike/scooter
    - percentiles aren't also necessarily the best here -- data is ordinal, not continuous; need more robust metric for number of people
- possibly go into more detail with number of children/non-household members as well

### traveler details (age, disability status, ability to drive)

In [ ]:
# traveler ability to drive

In [ ]:
df["can_drive"].value_counts()

In [ ]:
# bar plot of can drive vs can't drive
# most people can drive
# does htis include car ownership or just license?
sns.countplot(x="can_drive", data=df[df["can_drive"] != "Missing"])

In [ ]:
# side by side bar plot of the % of epople in each mode who can drive/can't drive
# relatively uniform aside from transit -- people who can drive have a lesser tendency to not use transit & vice versa
# transit may directly substitute car trips while other modes simply complement them
fig, ax = plot_multi_barplot(df[df["can_drive"] != "Missing"], "can_drive", "mode")

In [ ]:
# number of vehicles -- only accounts for household, not whether an individual can drive (has license)

In [ ]:
# get into a numeric form
df["num_vehicles_numeric"] = np.where(df["num_vehicles"] == "8+", 10, df["num_vehicles"].replace("8+", 9).astype(int))

In [ ]:
sns.countplot(x="num_vehicles_numeric", data=df)

In [ ]:
# analysis on number of vehicles
fig, ax = plot_multi_barplot(df, "num_vehicles_numeric", "mode", figsize=(10, 5))

In [ ]:
# age analysis

In [ ]:
df["age"].value_counts()

In [ ]:
# merge the two different labeling schemes
df["age_cleaned"] = df["age"].apply(lambda x: '-'.join(x.split(" to ")))
df["age_cleaned"] = np.where(df["age_cleaned"] == "5-15", "05-15", df["age_cleaned"])
df["age_cleaned"] = np.where(df["age_cleaned"] == "Under 5", "00-05", df["age_cleaned"])

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
sns.countplot(x="age_cleaned", data=df, ax=ax)

In [ ]:
# side by side bar plot of the % of trips of each mode that were made by certain age groups
# generally, car usage is less dominant from 18-44, after which car becomes more dominant
# transit usage decays from a peak in 25-34
plot_multi_barplot(df, "age_cleaned", y="mode", figsize=(15, 7), order=df["age_cleaned"].value_counts().sort_index().index)

In [ ]:
# age - can't really rule out anything
    # maybe old people can't bike
# one record for each person id -- both

In [ ]:
# disability

In [ ]:
df["disability"].value_counts()

In [ ]:
# clean all missing things into one NA value
df["disability_cleaned"] = np.where(df["disability"].isin(["No", "Yes"]), df["disability"], "NA")

In [ ]:
df["disability_cleaned"].value_counts()

In [ ]:
sns.countplot(x="disability_cleaned", data=df[df["disability_cleaned"] != "NA"])

In [ ]:
# side by side bar plot of the % of trips via each mode who do/do not have disabilities
# similar trend to those without cars -- those with disabilities tend to travel via transit, with everything else uniform
plot_multi_barplot(df[df["disability_cleaned"] != "NA"], "disability_cleaned", "mode")

takeaways
- those who cannot drive/are disabled have a greater likelihood of choosing transit, although not very significantly
    - mirrored by vehicle count somewhat, but this variable is less individualized than these
    - there is likely some overlap between these groups
    - with these variables, all other modes are uniform -- there is something unique about transit in catering to these people
- the relationship between age & mode choice/preferences are complex, but there definitely do seem to be patterns

future analysis
- possibly explore more personal variables
- can explore the combination of personal variables with other ones
    - e.g., people with disabilities that have multiple people in their household/travel with multiple people often

### bike lts share

**note:** this relies heavily on the assumptions made in bike routing and more reflect the design deatils of that optimization; any cutoff isn't based on data

In [ ]:
bike["lts1"] = bike["distance_meters_1"] / bike["distance_meters"]
bike["lts2"] = bike["distance_meters_2"] / bike["distance_meters"]
bike["lts3"] = bike["distance_meters_3"] / bike["distance_meters"]
bike["lts4"] = bike["distance_meters_4"] / bike["distance_meters"]

In [ ]:
df[df["mode"] == "Bike/Scooter"]["distance"] - bike.set_index('trip_id').loc[df[df["mode"] == "Bike/Scooter"].trip_id]['distance_meters']

In [ ]:
bike_lts_share = bike.set_index("trip_id").loc[df[df["mode"] == "Bike/Scooter"].trip_id][["lts1", "lts2", "lts3", "lts4", "distance_meters_1", "distance_meters_2", "distance_meters_3", "distance_meters_4"]]

In [ ]:
# distribution of % of trip taken in lts1, very right-heavy
# 0.05 percentile is effectively the 95th here
plot_density(bike_lts_share["lts1"], percentile=0.05)

In [ ]:
# distribution of distance taken in lts1, very left-heavy
plot_density(bike_lts_share["distance_meters_1"])

In [ ]:
# distribution of % of trip taken in lts2, left-heavy
plot_density(bike_lts_share["lts2"])

In [ ]:
# distribution of distance of trip taken in lts2
plot_density(bike_lts_share["distance_meters_2"], percentile=0.95)

In [ ]:
# distribution of % of trip taken in lts3
# like lts2 but more left-heavy
plot_density(bike_lts_share["lts3"])

In [ ]:
# distribution of distances traveled in lts3
plot_density(bike_lts_share["distance_meters_3"], percentile=0.95)

In [ ]:
# distribution of % of trip taken in lts4
# like lts3 but more left-heavy
plot_density(bike_lts_share["lts4"])

In [ ]:
# distribution of amount of trip taken in lts1
plot_density(bike_lts_share["distance_meters_4"], percentile=0.95)

In [ ]:
# lts % vs. temperature

In [ ]:
# merge temperature in
bike_lts_share["temperature"] = df.set_index("trip_id").loc[bike_lts_share.index]["temperature"]

In [ ]:
# scatter plot of temperature vs lts share
# generally, with lower temperatures, tend to cluster to lts 1 at the cost of everything else
# NOTE: this is based off rerouting; not necessarily canonical, although the rerouting is relatively accurate for biking
fig, (ax1, ax2) = plt.subplots(2, 2, figsize=(10, 8))
sns.scatterplot(x="lts1", y="temperature", data=bike_lts_share, ax=ax1[0])
sns.scatterplot(x="lts2", y="temperature", data=bike_lts_share, ax=ax1[1])
sns.scatterplot(x="lts3", y="temperature", data=bike_lts_share, ax=ax2[0])
sns.scatterplot(x="lts4", y="temperature", data=bike_lts_share, ax=ax2[1])

In [ ]:
# car vs bike lts 3 + 4

In [ ]:
bike["high_stress"] = bike["distance_meters_3"] + bike["distance_meters_4"]
bike["high_stress_pct"] = bike["high_stress"] / bike["distance_meters"]

In [ ]:
# can see that bike trips have more high stress magnitude distance
fig, ax = plt.subplots(figsize=(15, 10))
sns.histplot(0.000621371 * bike.set_index("trip_id").loc[df[df["mode"] == "Car"]["trip_id"].values]["high_stress"], ax=ax, stat="density", kde=True, label="car")
sns.histplot(0.000621371 * bike.set_index("trip_id").loc[df[df["mode"] == "Bike/Scooter"]["trip_id"].values]["high_stress"], ax=ax, stat="density", kde=True, label="bike")
ax.set_ylim(0, 10)
ax.set_xlim(0, 6)
plt.legend()

In [ ]:
# the above pattern may be due to the fact that car trips are just longer
fig, ax = plt.subplots(figsize=(15, 10))
sns.histplot(df[df["mode"] == "Car"]["distance"], ax=ax, stat="density", kde=True, label="car")
sns.histplot(df[df["mode"] == "Bike/Scooter"]["distance"], ax=ax, stat="density", kde=True, label="bike")
ax.set_xlim(0, 100)
plt.legend()

In [ ]:
# still observed even when car trips are filtered to the ones within the distance cutoff
fig, ax = plt.subplots(figsize=(15, 10))
sns.histplot(0.000621371 * bike.set_index("trip_id").loc[df[df["mode"] == "Car"]["trip_id"].values].query("distance_meters < 18500")["high_stress"], ax=ax, stat="density", kde=True, label="car")
sns.histplot(0.000621371 * bike.set_index("trip_id").loc[df[df["mode"] == "Bike/Scooter"]["trip_id"].values]["high_stress"], ax=ax, stat="density", kde=True, label="bike")
ax.set_xlim(0, 2)
ax.set_ylim(0, 10)
plt.legend()

In [ ]:
# when doing it by percent in high stress locations, seems to even out a bit more
fig, ax = plt.subplots(figsize=(15, 10))
sns.histplot(bike.set_index("trip_id").loc[df[df["mode"] == "Car"]["trip_id"].values]["high_stress_pct"], ax=ax, stat="density", kde=True, label="car")
sns.histplot(bike.set_index("trip_id").loc[df[df["mode"] == "Bike/Scooter"]["trip_id"].values]["high_stress_pct"], ax=ax, stat="density", kde=True, label="bike")
ax.set_ylim(0, 20)
plt.legend()

takeaways

- lts % is a strong measure of biking feasibility -- the vast majoirty of trips should be done mostly (w.r.t distance) on lts 1 paths
- lts distances generally cluster towards 0 as the lts increases
- as lts increases, the share of the distance on that lts generally decreases
- lts 1 share increases with lower temperatures

future analysis
- elevation change

### day of week

In [ ]:
# could be meaningful analysis but not necessarily related to feasibility

In [ ]:
df["travel_dow"].value_counts()

In [ ]:
sns.countplot(x="travel_dow", data=df)

In [ ]:
# mode share of car by day of week
# generally flat
plot_mode_barplot_pct_total(df, "travel_dow", "Car", func=lambda x: x / df["travel_dow"].value_counts())

In [ ]:
# mode share of transit by day of week
# sharp dip in saturday/sunday -- loss in service
plot_mode_barplot_pct_total(df, "travel_dow", "Transit", func=lambda x: x / df["travel_dow"].value_counts())

In [ ]:
# mode share of walking by day of week
# also generally stable
plot_mode_barplot_pct_total(df, "travel_dow", "Walk", func=lambda x: x / df["travel_dow"].value_counts())

In [ ]:
# mode share of biking by day of week
# mostly stable, but with dips in the middle of the week
plot_mode_barplot_pct_total(df, "travel_dow", "Bike/Scooter", func=lambda x: x / df["travel_dow"].value_counts())

day of week takeaways

- mode share does seem to change with the days of the week
- walk/driving is day-invariant while biking/transit are more dependent

### purpose

In [ ]:
# merging some closely related categories
df["purpose_cleaned"] = df["d_purpose_category"]
df["purpose_cleaned"] = np.where(df["purpose_cleaned"].isin(["School", "School-related"]), "School", df["purpose_cleaned"])
df["purpose_cleaned"] = np.where(df["purpose_cleaned"].isin(["Work", "Work-related"]), "Work", df["purpose_cleaned"])
df["purpose_cleaned"] = np.where(df["purpose_cleaned"].isin(["Errand/Other", "Errand"]), "Errand", df["purpose_cleaned"])
df["purpose_cleaned"] = np.where(df["purpose_cleaned"].isin(["Shop", "Shopping"]), "Shop", df["purpose_cleaned"])
df["purpose_cleaned"] = np.where(df["purpose_cleaned"].isin(["Missing: Non-response", "Missing: Skip logic", "Not imputable"]), "Missing", df["purpose_cleaned"])

In [ ]:
# overall distribution
fig, ax = plt.subplots(figsize=(20, 5))
sns.countplot(x="purpose_cleaned", data=df[df["purpose_cleaned"] != "Missing"], ax=ax)
plt.xticks(rotation=45)

In [ ]:
# purpose segmented by mode -- % of all mode trips in each purpose category
# notable features: social/recreation is predominantly walking, errands/shopping/escorts are mostly car
fig, ax = plot_multi_barplot(df[df["purpose_cleaned"] != "Missing"], "purpose_cleaned", "mode", figsize=(20, 5))
plt.xticks(rotation=45)

purpose takeaways

- some purposes are dominated by certain modes
    - things like going home are relatively mode-agnostic
    - going to work draws a higher proportion of transit trips
    - shopping draws a higher proportion of car trips
    - vast majority of walk trips are made for social/recreational trips

future analysis
- detailed purposes? or some recoding

### generalized weight (bike/walk)

**note**: these more reflect the routing metholodology as opposed to the real behavior observed in the data

In [ ]:
# bike weight summary
bike["weight"].describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# bike generalized weight distribution
plot_density(bike["weight"])

In [ ]:
# walking generalized weights summary
walk["weight"].describe()

In [ ]:
# walking generalized weights distribution
plot_density(walk["weight"])

takeaways

- lower weights are generally favored
- not sure if weight scales are transferred across modes
- this relies on rerouting analyses and not the ground truth of the surveys -- possibly biased/not reflective of real choices

### transit trip specifics -- transfers, legs, walking time

In [ ]:
# use the raw tbi data, which contains canonical details on egress/access and number of legs

In [ ]:
wave1_trips = pd.read_csv(data_dir + "/Data/TBI Wave 1 Dataset 20200630/trip.csv")
wave2_trips = pd.read_csv(data_dir + "/Data/Wave 2 Data Deliverable/trip.csv")
raw_trips = pd.concat([wave1_trips, wave2_trips]).set_index("trip_id")
raw_trips

In [ ]:
# number of legs

In [ ]:
# summary statistics of leg count
# should theoretically correspond to number of transfers, excluding the cases where extra trips are inputted into the linked trip overall
df[df["mode"] == "Transit"]["num_unlinked_trips"].describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# distribution of leg count
# 3 is reasonably most common -- access, transit, egress
# the majority of trips are also done in 1 trip; this is likely due to the access/egress parts of the trip not being recorded correctly
sns.countplot(x="num_unlinked_trips", data=df[df["mode"] == "Transit"])

In [ ]:
# transfer count

In [ ]:
# create a transit trip subset and extract trip_ids into a list
transit_trips = df[df["mode"] == "Transit"]
transit_trips["trip_ids"] = transit_trips["trip_id"].apply(lambda x: literal_eval(x))
transit_trips

In [ ]:
def get_num_transfers(trips, wave):
    res = 0
    for trip in trips:
        # count the number of trips whose destination purposes are to change mode
        if wave == 2 and raw_trips.loc[trip]["d_purpose_category"] == 11 or \
            wave == 1 and raw_trips.loc[trip]["d_purpose_category"] == 10:
            res += 1
    return res

In [ ]:
# calculate number of transfers
# NOTE: this only approximates the amount of transfers; some linked trips may include egress/access trip as an actual transfer
transit_trips["transfers"] = transit_trips.apply(lambda x: get_num_transfers(x["trip_ids"], x["wave"]), axis=1)

In [ ]:
# the most common transfer count is 0; majority of trips have 2 or less transfers
sns.countplot(x="transfers", data=transit_trips)

In [ ]:
# 95% of trips are 3 transfers or less
transit_trips["transfers"].describe(percentiles=[0.25, 0.5, 0.75, 0.8, 0.85, 0.9, 0.95])

In [ ]:
# non-transit time

In [ ]:
def calc_nontransit_time(trips, wave):
    res = 0
    for trip in trips:
        curr = raw_trips.loc[trip]
        if wave == 1 and curr["mode_type"] not in [1, 3, 4] and (curr["o_purpose_category_imputed"] != 10 or curr["d_purpose_category"] != 10):
            res += curr["duration"]
        elif wave == 2 and curr["mode_type"] not in [12, 13, 14] and (curr["o_purpose_category"] != 11 or curr["d_purpose_category"] != 11):
            res += curr["duration"]
    return res

In [ ]:
# NOTE: this is an approximation of non-transit time (minutes)
# some trips exclude the access/egress subtrips (could be negligible), so this is likely an underestiamte
# this is also mode-independent, which is a problem
transit_trips["nontransit_time"] = transit_trips.apply(lambda x: calc_nontransit_time(x["trip_ids"], x["wave"]), axis=1)

In [ ]:
# nontransit time (minutes) summary
transit_trips["nontransit_time"].describe(percentiles=[0.25, 0.5, 0.75, 0.95])

In [ ]:
# nontransit time (minutes) density plot
# very left-heavy
plot_density(transit_trips["nontransit_time"])

In [ ]:
# distance to a stop (to begin the trip)

In [ ]:
def get_dist_to_stop(trips, wave):
    target = raw_trips.loc[trips[0]]
    if target["d_purpose_category"] in [10, 11] and (wave == 1 and target["mode_type"] not in [1, 3, 4] or wave == 2 and target["mode_type"] not in [12, 13, 14]):
        if not np.isnan(target["distance"]):
            return target["distance"]
    return 0.1

In [ ]:
# NOTE: this assumes that for trips that don't have a specified access trip (or have one that is 0 distance), the transit stop is 0.1 miles away (essentially negligible distance)
transit_trips["dist_to_stop"] = transit_trips.apply(lambda x: get_dist_to_stop(x["trip_ids"], x["wave"]), axis=1)

In [ ]:
transit_trips["dist_to_stop"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
fig, ax = plot_density(transit_trips["dist_to_stop"])

takeaways

- all these things (including num_unlinked_trips) are approximations using raw trip data
    - can likely be improved but are reasonable approximations right now
    - these approximations are limited by the robustness of the data
- these factors seem to have significant influence on the calculus of individuals 
- vast majority of trips are 2 or less transfers; 95% are less than 3
- nontransit time is bounded by 21 minutes at the 95% percentile
    - very left-heavy

future analysis:
- better approximation techniques
- fix the linked trip logic (go by close time proximity, matching purposes)

### timing analysis

proposal:

- feasible mode shift means all at least all fixed things in a complete tour can still be done (can work around non-fixed, discretionary legs of a tour as needed)
- probable mode shift means all parts of a tour can still be done (including the non-fixed, discretionary legs of a tour)

In [ ]:
# fixed things: work, work-related, school, school-related
fixed_purposes = ["Work", "Work-related", "Escort", "School", "School-related"]
# feasible mode shift means all fixed things can still be done
# probable mode shift means all parts of a tour can still be done (e.g., shopping, meal, social/recreation)
df["d_purpose_category"].value_counts()

In [ ]:
# typical number of fixed trips in a daily activity pattern
# 1 is the vast majority of fixed purpose trips in a given tour
df.groupby(["wave", "person_id", "travel_date"]).apply(lambda x: x["d_purpose_category"].isin(fixed_purposes).sum()).describe()

In [ ]:
# fixed purpose counts in a complete tour are generally 0-2, with some major outliers
# high outliers are generally due to working in high mobility occupation (e.g., delivery drivers)
df.groupby(["wave", "person_id", "travel_date"]).apply(lambda x: x["d_purpose_category"].isin(fixed_purposes).sum()).value_counts()

In [ ]:
def convert_to_minutes(str):
    hours, minutes, seconds = [int(x) for x in str.split(":")]
    return hours * 60 + minutes + seconds / 60

def evaluate_timing(df, alt_mode_times, type="feasible"):
    if type == "feasible":
        return df.groupby(["wave", "person_id", "travel_date"]).apply(lambda x: evaluate_feasible_timing(x, alt_mode_times))
    elif type == "probable":
        return df.groupby(["wave", "person_id", "travel_date"]).apply(lambda x: evaluate_probable_timing(x, alt_mode_times))
    else:
        raise ValueError("invalid type")

def evaluate_feasible_timing(chunk, alt_mode_times: str):
    # if there is only an inbound and outbound trip, don't need to worry about timing
    if len(chunk) == 2:
        return True
    leg_starts = chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values
    ref = leg_starts[0]
    # start times, starting by 0 and accounting for midnight wraparound with the mod function, for each leg of the complete tour
    leg_starts = [(x - ref) % 1440 for x in chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values]
    leg_durations = chunk["duration"].values
    # calculate end times relative to the start times using the duration category
    leg_ends = [(x + y) for (x, y) in zip(leg_starts, leg_durations)]
    # these are arrays indicating whether each leg of the complete tour is a fixed arrival/departure
    fixed_arrivals = chunk["d_purpose_category"].isin(fixed_purposes).values
    fixed_departures = chunk["o_purpose_category"].isin(fixed_purposes).values
    
    # sanity check; if the atlernative time for any of the legs can't be found, return False (means can't route it feasibly, usually for transit)
    # do this as preprocessing
    # if ~(chunk["trip_id"].isin(alt_mode_times.index).any()):
    #     return False
    # alternative durations for each of the legs
    alt_durations = chunk[alt_mode_times].values # make it a col in the dataframe to simplify things
    # if any invalid durations, means routing wasn't possible (generally only for transit)
    # return true since this isn't due to timing issues
    if -1 in alt_durations:
        return True
    
    # return true if there aren't any fixed things to work around
    # also return true if there is only one fixed thing--all trips of these kind can be boiled down to traveling to the fixed thing and traveling back if all non-fixed, discretionary trips are omitted; which can be scheduled around feasibly
    if fixed_arrivals.sum() <= 1:
        return True
    
    # keeps track of a previous fixed arrival trip to compare against a current one (see whether they overlap)
    prev_fixed_arrival = -1
    for i in range(len(chunk)):
        if fixed_arrivals[i]: # if current trip is fixed arrival
            if prev_fixed_arrival != -1: # if there exists some previous fixed arrival trip
                if leg_ends[i] - alt_durations[i] < leg_ends[prev_fixed_arrival]: # if there is an overlap between this trip and the pregvious fixed arrival trip, not feasible
                    return False
            prev_fixed_arrival = i # update previous fixed arrival trip
            
    # basically the same as the above, except work backwards since we want to consider whether the next trip would overlap
    next_fixed_departure = -1
    for i in range(len(chunk)-1, -1, -1):
        if fixed_departures[i]:
            if next_fixed_departure != -1:
                if leg_starts[i] + alt_durations[i] > leg_starts[next_fixed_departure]:
                    return False
                next_fixed_departure = i

    # feasible if nothing is weird
    return True

def evaluate_probable_timing(chunk, alt_mode_times: str):
    # if there is only an inbound and outbound trip, don't need to worry about timing
    if len(chunk) == 2:
        return True
    leg_starts = chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values
    ref = leg_starts[0]
    # start times, starting by 0 and accounting for midnight wraparound with the mod function, for each leg of the complete tour
    leg_starts = [(x - ref) % 1440 for x in chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values]
    leg_durations = chunk["duration"].values
    # calculate end times relative to the start times using the duration category
    leg_ends = [(x + y) for (x, y) in zip(leg_starts, leg_durations)]
    # these are arrays indicating whether each leg of the complete tour is a fixed arrival/departure
    fixed_arrivals = chunk["d_purpose_category"].isin(fixed_purposes).values
    fixed_departures = chunk["o_purpose_category"].isin(fixed_purposes).values
    
    # sanity check; if the atlernative time for any of the legs can't be found, return False (means can't route it feasibly, usually for transit)
    # do this as preprocessing
    # if ~(chunk["trip_id"].isin(alt_mode_times.index).any()):
    #     return False
    # alternative durations for each of the legs
    alt_durations = chunk[alt_mode_times].values # make it a col in the dataframe to simplify things
    if -1 in alt_durations:
        return True
    
    # need to omit this for probable -- could have other, non-fixed trips in a tour
    # if fixed_arrivals.sum() <= 1:
    #     return True
    
    for i in range(1, len(chunk) - 1):# can always neglect trip 0 (just leave earlier) and the last trip (just arrive later)
        # if the current trip is fixed arrival
        if fixed_arrivals[i]:
            # for probable, consider ALL adjacent trips; not just adjacent trips that are fixed
            # if you would need to start before the last trip finished, not feasible
            if leg_ends[i] - alt_durations[i] < leg_ends[i - 1]: 
                return False
        # if the current trip is fixed departure
        if fixed_departures[i]:
            # if you would arrive after the next trip began, not feasible
            if leg_starts[i] + alt_durations[i] > leg_starts[i + 1]: 
                return False

    # feasible if nothing is weird
    return True

In [ ]:
# need to add the target mode alt durations to the dataframe directly
# NOTE: this expects invalid durations to be set to -1 for accurate results
df["alt_walk_time"] = (walk.set_index("trip_id").loc[df["trip_id"]]["duration_seconds"] / 60).values
feasible = evaluate_timing(df, "alt_walk_time", "feasible")
probable = evaluate_timing(df, "alt_walk_time", "probable")

In [ ]:
# if everybody were to switch to walking, 88.5% would be feasible given timing constraints and 69.1% would be probable given timing constraints
# see beginning of section for clarification on what feasible/probable means in the context of timing
print(feasible.sum() / len(feasible))
print(probable.sum() / len(probable))

takeaways:

- a non-insignificant number of all trips are constrained by timing
    - varies between feasible/probable

### cold start analysis

- a cold start is defined as a car trip occurring more than 15 minutes after the previous one
    - problematic since cold starts result in more emissions

In [ ]:
COLD_START_THRESHOLD = 15

def get_num_cold_starts(chunk):
    leg_starts = chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values
    ref = leg_starts[0]
    # start times, starting by 0 and accounting for midnight wraparound with the mod function, for each leg of the complete tour
    leg_starts = [(x - ref) % 1440 for x in chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values]
    leg_durations = chunk["duration"].values
    # calculate end times relative to the start times using the duration category
    leg_ends = [(x + y) for (x, y) in zip(leg_starts, leg_durations)]
    modes = chunk["mode"].values
    
    cold_starts = 0
    prev_end = -1
    # iterate over all trips in a complete tour
    for i in range(len(leg_starts)):
        # if the current leg mode i car
        if modes[i] == "Car":
            # if there wasn't a previous or the difference between the end of the last car trip and the beginning of this car trip is more than 15 minutes, it is a cold start
            if prev_end == -1 or leg_starts[i] - leg_ends[i] > COLD_START_THRESHOLD:
                cold_starts += 1
            # update previous end of car trip
            prev_end = leg_ends[i]
    return cold_starts            

In [ ]:
x = df.groupby(["wave", "person_id", "travel_date"]).apply(lambda x: get_num_cold_starts(x))

In [ ]:
x.sum()